# 07 — Export OOF evidence for the 0.943 ensemble (CPU only)

The previous inventory found small public OOF files. This notebook copies **only those evidence files and the competition train tables** into its output so they can be audited offline. It does not train, infer on test, tune weights, or submit.

Run as a **new** Kaggle notebook with Accelerator **None** and Internet **Off**. Check that the same public inputs from notebook 06 are attached. Save & Run All; send the Output ZIP.


In [ ]:
from pathlib import Path
import csv, hashlib, json, shutil

ROOT = Path('/kaggle/input')
OUT = Path('/kaggle/working/oof_evidence')
if not ROOT.is_dir():
    raise FileNotFoundError('Run this notebook on Kaggle; /kaggle/input is missing')
OUT.mkdir(parents=True, exist_ok=True)
MAX_FILE_BYTES = 25_000_000
MAX_TOTAL_BYTES = 40_000_000
SOURCES = [
    'datasets/antoinegg1/rsna-knee-e11-diverse-heads-v20/v52_e11_oof.csv',
    'datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15/v52_oof.csv',
    'datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15/public_oof_diagnostic.json',
    'datasets/pilkwang/rsna-knee-weights/oof.npz',
    'datasets/pilkwang/rsna-knee-weights/manifest.json',
    'notebooks/sofiaanjenje/rsna-knee-e11-train/rsna_rad_e11/v52_e11_oof.csv',
    'notebooks/sofiaanjenje/rsna-knee-e13-train/rsna_rad_e11/v52_e11_oof.csv',
]
competition_roots = [p for p in (ROOT / 'competitions').glob('*') if p.is_dir()]
competition_roots += [p for p in ROOT.glob('*') if p.is_dir() and (p / 'train.csv').is_file()]
for name in ('train.csv', 'train_series.csv'):
    hits = sorted({str(p / name) for p in competition_roots if (p / name).is_file()})
    if len(hits) == 1:
        SOURCES.append(str(Path(hits[0]).relative_to(ROOT)))
    else:
        print(f'WARNING: expected exactly one competition {name}, found {len(hits)}: {hits[:5]}')
records, skipped, total = [], [], 0
for rel in SOURCES:
    src = ROOT / rel
    if not src.is_file():
        raise FileNotFoundError(f'Missing attached input: {src}')
    size = src.stat().st_size
    if size > MAX_FILE_BYTES or total + size > MAX_TOTAL_BYTES:
        if rel.startswith('competitions/'):
            skipped.append({'path': rel, 'bytes': size, 'reason': 'optional competition table exceeds export size guard'})
            print(f'WARNING: skipped optional large table {rel} ({size:,} bytes)')
            continue
        raise RuntimeError(f'Required OOF evidence exceeds export size guard: {rel}, {size} bytes')
    dst = OUT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(src, dst)
    digest = hashlib.sha256(dst.read_bytes()).hexdigest()
    record = {'path': rel, 'bytes': size, 'sha256': digest}
    if dst.suffix.lower() == '.csv':
        with dst.open(newline='', encoding='utf-8-sig') as fh:
            reader = csv.reader(fh)
            record['columns'] = next(reader, [])
            record['data_rows'] = sum(1 for _ in reader)
    records.append(record)
    total += size
    print(f'{rel}: {size:,} bytes; {record.get("data_rows", "non-CSV")} rows')
(OUT / 'export_manifest.json').write_text(json.dumps({
    'purpose': 'Offline audit of public OOF provenance, IDs, labels, and fold independence; no weight selection yet',
    'files': records, 'skipped': skipped, 'total_bytes': total,
}, indent=2), encoding='utf-8')
print(f'Exported {len(records)} evidence files; {total:,} bytes. Download this notebook Output ZIP.')
